# NetWeaver SRE — Unsloth + GRPO Training

Train a small LLM (default `Qwen/Qwen2.5-0.5B-Instruct`, swappable for `unsloth/gpt-oss-20b` on bigger GPUs) to be an autonomous Site Reliability Engineer.

**Pattern follows the official [OpenEnv × Unsloth tutorial](https://github.com/meta-pytorch/OpenEnv/blob/main/tutorial/examples/unsloth_2048.ipynb)**:
* Unsloth `FastLanguageModel` + 4-bit + LoRA → fits free Colab T4
* Composed reward functions (3 separate signals) → rich GRPO learning signal
* In-process FastAPI server → no subprocess setup
* TrackIO live visualization

**Runtime: free Colab T4 GPU. ~50 GRPO steps in ~25–40 minutes.**

## 1. Clone the repo + install dependencies

In [ ]:
import os, importlib.util
if not os.path.isdir('netweaver-sre'):
    !git clone https://github.com/Shasidharyadav/netweaver-sre.git
%cd netweaver-sre

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec('torch') is None or 'COLAB_' in ''.join(os.environ.keys()):
    try: import numpy; get_numpy = f'numpy=={numpy.__version__}'
    except: get_numpy = 'numpy'
    !uv pip install -qqq \
        'torch>=2.4.0' 'triton>=3.0.0' {get_numpy} torchvision bitsandbytes 'transformers>=4.44.0' \
        'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo' \
        'unsloth[base] @ git+https://github.com/unslothai/unsloth' \
        trackio
elif importlib.util.find_spec('unsloth') is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps trl tokenizers unsloth unsloth_zoo
!pip install -qqq fastapi uvicorn requests httpx pillow datasets matplotlib

## 2. Boot the NetWeaver SRE environment server in-process

We launch the FastAPI server in a daemon thread so this notebook owns the env lifecycle. Using the in-process `TestClient` would also work, but the HTTP path is the same one the OpenEnv judges will exercise.


In [ ]:
import os, sys, time, threading, requests, uvicorn
ROOT = os.getcwd()
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('ENV_URL', 'http://0.0.0.0:8000')

from server.app import app

def _run():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=_run, daemon=True).start()
for _ in range(30):
    try:
        if requests.get('http://0.0.0.0:8000/health', timeout=1).status_code == 200:
            print('server ready'); break
    except Exception: time.sleep(0.5)
else:
    raise RuntimeError('server failed to start')

## 3. Sanity check the env (1 episode of T01)

In [ ]:
import requests, re
ENV = 'http://0.0.0.0:8000'

def post(p, j=None): return requests.post(f'{ENV}/{p}', json=j or {}).json()
def get(p): return requests.get(f'{ENV}/{p}').json()

post('set_level', {'task_level': 't01'})
obs = post('reset')['observation']
print('Alert:', obs['alert'])
node = re.search(r'node_\d+', obs['alert']).group(0)
step = post('step', {'action': {'command': 'DRAIN_TRAFFIC', 'target': node, 'value': None}})
print('Resolved:', step['observation'].get('resolved'), 'Reward:', step['reward'])

## 4. Load Qwen2.5-0.5B-Instruct with Unsloth (4-bit + LoRA)

Default model is `Qwen/Qwen2.5-0.5B-Instruct` (fits on free T4). To use GPT-OSS 20B, change `MODEL_NAME` to `unsloth/gpt-oss-20b` (needs L4 / A100).

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-0.5B-Instruct')
MAX_SEQ_LENGTH = 1024
LORA_RANK = 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    max_seq_length=MAX_SEQ_LENGTH,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Loaded:', MODEL_NAME)

## 5. Build the training dataset (prompts derived from real env resets)

In [ ]:
import random, json
from datasets import Dataset
from train_grpo import build_prompt, TASK_LEVELS, EnvAdapter

env = EnvAdapter(ENV)

rows = []
for _ in range(128):
    level = random.choice(TASK_LEVELS)
    env.set_level(level)
    obs = env.reset().get('observation', {})
    rows.append({'prompt': build_prompt(obs, level)})

dataset = Dataset.from_list(rows)
print('Dataset size:', len(dataset))
print('Example prompt (first 400 chars):')
print(dataset[0]['prompt'][:400], '...')

## 6. Pre-training evaluation (random baseline)

In [ ]:
from train_grpo import evaluate_model
before_rewards, before_diff = evaluate_model(env, model, tokenizer, episodes=10)
before_avg = sum(before_rewards) / len(before_rewards)
print(f'Before-training avg reward: {before_avg:.3f}')
print('By difficulty:', before_diff)

## 7. GRPO training with composed reward functions

Three separate reward signals are passed to GRPOTrainer:
1. `reward_action_parses` — +1 if completion parses to a valid action
2. `reward_correct_command` — +2 if the issued command is valid for the fault type
3. `reward_episode_resolution` — the big signal: full multi-step rollout → grader rubric

This pattern matches the OpenEnv × Unsloth tutorial (`function_works`, `no_cheating`, `strategy_succeeds`).

Watch the `reward` column in the table below — it should climb from ~-1 to ~+5 over training.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from train_grpo import (reward_action_parses, reward_correct_command,
                       reward_episode_resolution, MAX_TRAIN_STEPS)

# Inject runtime context
reward_episode_resolution.env = env
reward_episode_resolution.model = model
reward_episode_resolution.tokenizer = tokenizer

training_args = GRPOConfig(
    temperature=1.0,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    optim='adamw_8bit',
    logging_steps=1,
    per_device_train_batch_size=2,
    num_generations=2,
    gradient_accumulation_steps=1,
    max_prompt_length=900,
    max_completion_length=128,
    max_steps=int(os.environ.get('MAX_TRAIN_STEPS', '50')),
    save_steps=200,
    report_to='trackio',
    output_dir='outputs',
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_action_parses, reward_correct_command, reward_episode_resolution],
    args=training_args,
    train_dataset=dataset,
)
trainer.train()

## 8. Post-training evaluation

In [ ]:
after_rewards, after_diff = evaluate_model(env, model, tokenizer, episodes=10)
after_avg = sum(after_rewards) / len(after_rewards)
print(f'Before-training avg reward: {before_avg:.3f}')
print(f'After-training  avg reward: {after_avg:.3f}')
print(f'Δ = {after_avg - before_avg:+.3f}')

## 9. Persist results + render plots

In [ ]:
import json, time
from train_grpo import TRAIN_REWARDS, save_plots
results = {
    'env_url': ENV,
    'model_name': MODEL_NAME,
    'max_train_steps': training_args.max_steps,
    'training_rewards': TRAIN_REWARDS[:training_args.max_steps],
    'before_rewards': before_rewards,
    'after_rewards': after_rewards,
    'difficulty_breakdown': {'before': before_diff, 'after': after_diff},
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'source': 'grpo_real_run',
}
with open('training_results.json', 'w') as f:
    json.dump(results, f, indent=2)
save_plots(before_rewards, after_rewards, before_diff, after_diff)
print('Saved: training_results.json + server/assets/*.png')

## 10. Inline visualization

In [ ]:
from IPython.display import Image, display
for p in ['server/assets/reward_curve.png',
         'server/assets/before_after.png',
         'server/assets/difficulty_breakdown.png']:
    if os.path.exists(p): display(Image(p))

## 11. (Optional) Push the trained LoRA to the Hub

In [ ]:
# from huggingface_hub import login
# from google.colab import userdata
# login(token=userdata.get('HF_TOKEN'))
# model.push_to_hub('YOUR-USERNAME/netweaver-sre-qwen-grpo')
# tokenizer.push_to_hub('YOUR-USERNAME/netweaver-sre-qwen-grpo')